In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np

In [2]:
from tensorflow.keras.applications import MobileNetV2

In [4]:
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 5s 1us/step


In [5]:
base_model.trainable = False

In [6]:
print(base_model.trainable)

False


In [7]:
model_mobilenet = keras.Sequential([
    
    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.3),

    layers.Dense(6, activation="softmax")
])

In [8]:
model_mobilenet.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [9]:
model_mobilenet.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,726 (9.24 MB)

 Trainable params: 164,742 (643.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [11]:
dataset_path = "Data/dataset-resized"

img_size = (224, 224)
batch_size = 32

In [12]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

Found 2527 files belonging to 6 classes.
Using 2022 files for training.


In [13]:
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

Found 2527 files belonging to 6 classes.
Using 505 files for validation.


In [14]:
print(train_ds)
print(val_ds)

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>


In [15]:
history_mobilenet = model_mobilenet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 85s 1s/step - accuracy: 0.5005 - loss: 1.3314 - val_accuracy: 0.5822 - val_loss: 1.0726
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - accuracy: 0.6330 - loss: 1.0056 - val_accuracy: 0.6119 - val_loss: 1.0364
Epoch 3/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - accuracy: 0.6751 - loss: 0.8854 - val_accuracy: 0.5822 - val_loss: 1.0225
Epoch 4/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 80s 1s/step - accuracy: 0.7013 - loss: 0.8061 - val_accuracy: 0.6455 - val_loss: 0.9479
Epoch 5/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - accuracy: 0.7433 - loss: 0.7303 - val_accuracy: 0.6713 - val_loss: 0.8889
Epoch 6/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 81s 1s/step - accuracy: 0.7666 - loss: 0.6622 - val_accuracy: 0.6535 - val_loss: 0.8910
Epoch 7/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 75s 1s/step - accuracy: 0.7765 - loss: 0.6213 - val_accuracy: 0.6733 - val_loss: 0.8754
Epoch 8/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 82s 1s/step - accuracy: 0.7819 - loss: 0.5758 - val_accuracy: 0.6911 - val_loss:

In [16]:
model_mobilenet.save("mobilenet_baseline.keras")

In [17]:
base_model.trainable = True

In [18]:
print(base_model.trainable)

True


In [19]:
model_mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [20]:
history_finetune = model_mobilenet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 199s 2s/step - accuracy: 0.2864 - loss: 3.9661 - val_accuracy: 0.3644 - val_loss: 2.8904
Epoch 2/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 126s 2s/step - accuracy: 0.4975 - loss: 1.5862 - val_accuracy: 0.3446 - val_loss: 3.8337
Epoch 3/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 257s 4s/step - accuracy: 0.6212 - loss: 1.0440 - val_accuracy: 0.3782 - val_loss: 3.6584
Epoch 4/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 322s 5s/step - accuracy: 0.6771 - loss: 0.8694 - val_accuracy: 0.4040 - val_loss: 3.3040
Epoch 5/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 283s 4s/step - accuracy: 0.7206 - loss: 0.7565 - val_accuracy: 0.4337 - val_loss: 2.9146
